<a href="https://www.kaggle.com/code/sunainadas/titanic-xgboost?scriptVersionId=347751808" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
df=pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.shape

(891, 12)

In [4]:
#len(sorted(df["Ticket"].unique()))
sum(df["Ticket"].isna())

0

In [5]:
len(df["Cabin"])

891

In [6]:
df["Cabin"].isna().sum()

np.int64(687)

In [7]:
df=df.drop(["Ticket","Cabin","PassengerId"],axis=1)

In [8]:
df

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...,...
886,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,13.0000,S
887,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,30.0000,S
888,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,23.4500,S
889,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,30.0000,C


In [9]:
df = df.sample(frac=1).reset_index(drop=True)
df

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,"Berglund, Mr. Karl Ivar Sven",male,22.0,0,0,9.3500,S
1,1,3,"Niskanen, Mr. Juha",male,39.0,0,0,7.9250,S
2,1,1,"Robert, Mrs. Edward Scott (Elisabeth Walton Mc...",female,43.0,0,1,211.3375,S
3,0,3,"Bourke, Miss. Mary",female,NaN,0,2,7.7500,Q
4,0,2,"Mitchell, Mr. Henry Michael",male,70.0,0,0,10.5000,S
...,...,...,...,...,...,...,...,...,...
886,0,1,"Butt, Major. Archibald Willingham",male,45.0,0,0,26.5500,S
887,0,3,"Danbom, Mr. Ernst Gilbert",male,34.0,1,1,14.4000,S
888,1,3,"Lindqvist, Mr. Eino William",male,20.0,1,0,7.9250,S
889,0,3,"Bourke, Mrs. John (Catherine)",female,32.0,1,1,15.5000,Q


In [10]:
X=df.drop(["Survived"],axis=1)
X

,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,"Berglund, Mr. Karl Ivar Sven",male,22.0,0,0,9.3500,S
1,3,"Niskanen, Mr. Juha",male,39.0,0,0,7.9250,S
2,1,"Robert, Mrs. Edward Scott (Elisabeth Walton Mc...",female,43.0,0,1,211.3375,S
3,3,"Bourke, Miss. Mary",female,NaN,0,2,7.7500,Q
4,2,"Mitchell, Mr. Henry Michael",male,70.0,0,0,10.5000,S
...,...,...,...,...,...,...,...,...
886,1,"Butt, Major. Archibald Willingham",male,45.0,0,0,26.5500,S
887,3,"Danbom, Mr. Ernst Gilbert",male,34.0,1,1,14.4000,S
888,3,"Lindqvist, Mr. Eino William",male,20.0,1,0,7.9250,S
889,3,"Bourke, Mrs. John (Catherine)",female,32.0,1,1,15.5000,Q


In [11]:
y=df['Survived']

In [12]:
X=X.drop(['Name'],axis=1)

In [13]:
X.dtypes

Pclass        int64
Sex          object
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Embarked     object
dtype: object

In [14]:
X['Sex'].unique()

array(['male', 'female'], dtype=object)

In [15]:
X['Embarked'].unique()

array(['S', 'Q', 'C', nan], dtype=object)

# One Hot Encoding

In [16]:
X=pd.get_dummies(X,columns=["Sex",'Embarked','Pclass'],dtype=int)

In [17]:
X

,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Pclass_1,Pclass_2,Pclass_3
0,22.0,0,0,9.3500,0,1,0,0,1,0,0,1
1,39.0,0,0,7.9250,0,1,0,0,1,0,0,1
2,43.0,0,1,211.3375,1,0,0,0,1,1,0,0
3,NaN,0,2,7.7500,1,0,0,1,0,0,0,1
4,70.0,0,0,10.5000,0,1,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
886,45.0,0,0,26.5500,0,1,0,0,1,1,0,0
887,34.0,1,1,14.4000,0,1,0,0,1,0,0,1
888,20.0,1,0,7.9250,0,1,0,0,1,0,0,1
889,32.0,1,1,15.5000,1,0,0,1,0,0,0,1


In [18]:
(len(y)-sum(y))/sum(y)

1.605263157894737

# XGBoost

In [19]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

In [20]:
'''# Define parameter grid
param_grid = {
    'max_depth': [7,9,11],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'learning_rate': [0.01, 0.05, 0.1],
    'reg_lambda' : [0,1.0,10.0]
}

# Create XGBoost classifier
clf_xgb = xgb.XGBClassifier(n_estimators=500, objective='binary:logistic', 
                            random_state=42, scale_pos_weight=1.6, eval_metric='aucpr')

# Perform grid search
grid_search = GridSearchCV(estimator=clf_xgb, param_grid=param_grid, verbose=1,
                          cv=4)
grid_search.fit(X, y)'''

"# Define parameter grid\nparam_grid = {\n    'max_depth': [7,9,11],\n    'subsample': [0.6, 0.8, 1.0],\n    'colsample_bytree': [0.6, 0.8, 1.0],\n    'learning_rate': [0.01, 0.05, 0.1],\n    'reg_lambda' : [0,1.0,10.0]\n}\n\n# Create XGBoost classifier\nclf_xgb = xgb.XGBClassifier(n_estimators=500, objective='binary:logistic', \n                            random_state=42, scale_pos_weight=1.6, eval_metric='aucpr')\n\n# Perform grid search\ngrid_search = GridSearchCV(estimator=clf_xgb, param_grid=param_grid, verbose=1,\n                          cv=4)\ngrid_search.fit(X, y)"

In [21]:
'''n_estimators:200 - {'colsample_bytree': 0.8,
 'learning_rate': 0.1,
 'max_depth': 9,
 'reg_lambda': 10.0,
 'subsample': 1.0}'''
#running grid search again as learning rate very high

"n_estimators:200 - {'colsample_bytree': 0.8,\n 'learning_rate': 0.1,\n 'max_depth': 9,\n 'reg_lambda': 10.0,\n 'subsample': 1.0}"

In [22]:
'''{'colsample_bytree': 1.0,
 'learning_rate': 0.01,
 'max_depth': 7,
 'reg_lambda': 1.0,
 'subsample': 0.6}
'''
#grid_search.best_params_

"{'colsample_bytree': 1.0,\n 'learning_rate': 0.01,\n 'max_depth': 7,\n 'reg_lambda': 1.0,\n 'subsample': 0.6}\n"

In [23]:
#print("Best Validation Score:", grid_search.best_score_)
#Best Validation Score: 0.8260513877105805

In [24]:
# Define parameter grid
'''param_grid = {
    'max_depth': [7,9,11],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'learning_rate': [0.01, 0.05, 0.1],
    'reg_lambda' : [0,1.0,10.0]
}

# Create XGBoost classifier
clf_xgb = xgb.XGBClassifier(n_estimators=300, objective='binary:logistic', 
                            random_state=42, scale_pos_weight=1.6, eval_metric='aucpr')

# Perform grid search
grid_search = GridSearchCV(estimator=clf_xgb, param_grid=param_grid, verbose=1,
                          cv=4)
grid_search.fit(X, y)'''

"param_grid = {\n    'max_depth': [7,9,11],\n    'subsample': [0.6, 0.8, 1.0],\n    'colsample_bytree': [0.6, 0.8, 1.0],\n    'learning_rate': [0.01, 0.05, 0.1],\n    'reg_lambda' : [0,1.0,10.0]\n}\n\n# Create XGBoost classifier\nclf_xgb = xgb.XGBClassifier(n_estimators=300, objective='binary:logistic', \n                            random_state=42, scale_pos_weight=1.6, eval_metric='aucpr')\n\n# Perform grid search\ngrid_search = GridSearchCV(estimator=clf_xgb, param_grid=param_grid, verbose=1,\n                          cv=4)\ngrid_search.fit(X, y)"

In [25]:
#grid_search.best_params_
'''{'colsample_bytree': 1.0,
 'learning_rate': 0.01,
 'max_depth': 7,
 'reg_lambda': 0,
 'subsample': 0.6}'''

"{'colsample_bytree': 1.0,\n 'learning_rate': 0.01,\n 'max_depth': 7,\n 'reg_lambda': 0,\n 'subsample': 0.6}"

In [26]:
#print("Best Validation Score:", grid_search.best_score_)
#Best Validation Score: 0.8305205429644893

In [27]:
clf_xgb = xgb.XGBClassifier(n_estimators=300, objective='binary:logistic', 
                                random_state=42, scale_pos_weight=1.6, 
                                colsample_bytree= 1.0, eval_metric='aucpr',
                                learning_rate= 0.01, max_depth= 7,
                                  reg_lambda= 0, subsample= 0.6)

In [28]:
clf_xgb.fit(X,y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=1.0, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.01, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

# Training Dataset

In [29]:
df=pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
df=df.drop(["Ticket","Cabin","Name"],axis=1)
ID=df["PassengerId"]
X=df.drop(["PassengerId"],axis=1)
ID.head()

0    892
1    893
2    894
3    895
4    896
Name: PassengerId, dtype: int64

In [30]:
X=pd.get_dummies(X,columns=["Sex",'Embarked','Pclass'],dtype=int)
X.head()

,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Pclass_1,Pclass_2,Pclass_3
0,34.5,0,0,7.8292,0,1,0,1,0,0,0,1
1,47.0,1,0,7.0000,1,0,0,0,1,0,0,1
2,62.0,0,0,9.6875,0,1,0,1,0,0,1,0
3,27.0,0,0,8.6625,0,1,0,0,1,0,0,1
4,22.0,1,1,12.2875,1,0,0,0,1,0,0,1


In [31]:
y_pred=clf_xgb.predict(X)

In [32]:
submission = pd.DataFrame({
    'PassengerId': ID,
    'Survived': y_pred
})
submission.to_csv('submission.csv', index=False)